# Vulnerability Adaptation Test: Public Infrastructure

This notebook uses the reusable raster-explicit vulnerability workflow for a
public infrastructure test case.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd

from sovereign.flood import (
    BasinLossCurve,
    build_basin_curves,
    build_masked_vulnerability_scenario_curves,
    extract_sectoral_losses,
    run_simulation,
)


In [2]:
# USER CONFIG
model = "wri"
n_years = 5000
vulnerability_reduction = 0.50
mask_value = 1
mask_path = Path.cwd().parent / "outputs" / "flood" / "adaptation" / "urban_mask_23.tif"

# Target sector setup
target_sector_label = "Public"
target_exposure_name = "inf_pub_capstock.tif"
paired_public_component_name = "pub_nres"


In [3]:
# Paths and baseline inputs
root = Path.cwd().parent
flood_dir = root / "inputs" / "flood" / "maps"
exposure_dir = root / "outputs" / "exposure"
risk_map_dir = root / "outputs" / "flood" / "risk" / "maps"
adaptation_map_dir = root / "outputs" / "flood" / "risk" / "maps" / "adaptation"
basin_path = root / "outputs" / "boundaries" / "analysis_basins.gpkg"
risk_basin_path = root / "outputs" / "flood" / "risk" / "basins" / f"risk_basins_m-{model}.csv"
copula_path = root / "outputs" / "flood" / "dependence" / "copulas" / "copula_random_numbers.gzip"
vulnerability_path = root / "inputs" / "flood" / "vulnerability" / "jrc_depth_damage.csv"

adaptation_map_dir.mkdir(parents=True, exist_ok=True)

flood_dic = {
    5: "UGA_wri-flood_RP5.tif",
    10: "UGA_wri-flood_RP10.tif",
    25: "UGA_wri-flood_RP25.tif",
    50: "UGA_wri-flood_RP50.tif",
    100: "UGA_wri-flood_RP100.tif",
    250: "UGA_wri-flood_RP250.tif",
    500: "UGA_wri-flood_RP500.tif",
    1000: "UGA_wri-flood_RP1000.tif",
}

target_exposure_path = exposure_dir / target_exposure_name
paired_public_baseline_paths = {
    rp: str(risk_map_dir / f"WRI_{rp}_{paired_public_component_name}_cap_damages.tif")
    for rp in flood_dic
}

risk_data = pd.read_csv(risk_basin_path)
risk_data = risk_data.iloc[:, 1:]
risk_data["AEP"] = 1 / risk_data["RP"]
risk_data["Pr_L_AEP"] = np.where(risk_data["Pr_L"] == 0, 0, 1 / risk_data["Pr_L"])
risk_data.reset_index(drop=True, inplace=True)

copula_random_numbers = pd.read_parquet(copula_path).iloc[:n_years].copy()

risk_data.head()


,FID,GID_1,NAME,HB_L6,Pr_L,damages,adapted_damages,RP,Sector,AEP,Pr_L_AEP
0,0,UGA.3_1,Arua,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
1,1,UGA.47_1,Nebbi,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
2,2,UGA.27_1,Kitgum,1.060999e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
3,3,UGA.41_1,Moyo,1.061033e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
4,4,UGA.3_1,Arua,1.061033e+09,2.0,6160.324707,6160.324707,5,Public,0.2,0.5


In [4]:
# Vulnerability curves
vuln_df = pd.read_csv(vulnerability_path)
v_heights = vuln_df["flood_depth"].to_list()
v_inf = vuln_df["africa_infrastructure"].to_list()
v_inf_adapted = (pd.Series(v_inf) * (1 - vulnerability_reduction)).tolist()

baseline_damage_function = [v_heights, v_inf]
adapted_damage_function = [v_heights, v_inf_adapted]


## Build Scenario Curves

This reusable workflow:

1. builds adapted public infrastructure rasters inside the mask
2. combines them with unchanged public non-residential rasters
3. aggregates to basin risk rows
4. builds vulnerability scenario curves


In [5]:
baseline_curves: dict[int, BasinLossCurve] = build_basin_curves(risk_data)

vulnerability_scenario_curves, adapted_public_risk_df, adapted_public_total_paths = build_masked_vulnerability_scenario_curves(
    baseline_risk_df=risk_data,
    basin_path=str(basin_path),
    flood_map_lookup=flood_dic,
    flood_dir=str(flood_dir),
    target_exposure_path=str(target_exposure_path),
    mask_path=str(mask_path),
    baseline_damage_function=baseline_damage_function,
    adapted_damage_function=adapted_damage_function,
    target_sector_label=target_sector_label,
    output_dir=str(adaptation_map_dir),
    adapted_component_name_template=f"WRI_{{rp}}_pub_inf_cap_damages_vuln_reduction_{int(vulnerability_reduction * 100)}.tif",
    combined_sector_name_template=f"WRI_{{rp}}_pub_cap_damages_vuln_reduction_{int(vulnerability_reduction * 100)}.tif",
    baseline_component_paths_by_rp=paired_public_baseline_paths,
    mask_value=mask_value,
)

adapted_public_risk_df.head()


C:\Users\Mark.DESKTOP-UFHIN6T\anaconda3\envs\sovereign-risk\lib\site-packages\rasterstats\io.py:328: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


,FID,GID_1,NAME,HB_L6,Pr_L,damages,RP,Sector,AEP,Pr_L_AEP
0,0,UGA.3_1,Arua,1.061054e+09,2.0,0.000000,5,Public,0.2,0.5
1,1,UGA.47_1,Nebbi,1.061054e+09,2.0,0.000000,5,Public,0.2,0.5
2,2,UGA.27_1,Kitgum,1.060999e+09,2.0,0.000000,5,Public,0.2,0.5
3,3,UGA.41_1,Moyo,1.061033e+09,2.0,0.000000,5,Public,0.2,0.5
4,4,UGA.3_1,Arua,1.061033e+09,2.0,6160.324707,5,Public,0.2,0.5


## Run Simulation


In [ ]:
baseline_losses, vulnerability_adapted_losses = run_simulation(
    baseline_curves,
    vulnerability_scenario_curves,
    n_years,
    copula_random_numbers,
)


 24%|██████████████████▋                                                          | 1212/5000 [00:11<00:35, 105.63it/s]

In [ ]:
baseline_sectoral_loss = extract_sectoral_losses(baseline_losses, n_years)
vulnerability_sectoral_loss = extract_sectoral_losses(vulnerability_adapted_losses, n_years)

comparison_df = pd.DataFrame({
    "metric": ["GVA_loss", "CAP_dam", "AGR_loss", "MAN_loss", "SER_loss", "PUB_dam", "PRI_dam"],
    "baseline_aal": [baseline_sectoral_loss[c].mean() for c in ["GVA_loss", "CAP_dam", "AGR_loss", "MAN_loss", "SER_loss", "PUB_dam", "PRI_dam"]],
    "vulnerability_aal": [vulnerability_sectoral_loss[c].mean() for c in ["GVA_loss", "CAP_dam", "AGR_loss", "MAN_loss", "SER_loss", "PUB_dam", "PRI_dam"]],
})
comparison_df["aal_change"] = comparison_df["vulnerability_aal"] - comparison_df["baseline_aal"]
comparison_df["pct_change"] = np.where(
    comparison_df["baseline_aal"] != 0,
    100 * comparison_df["aal_change"] / comparison_df["baseline_aal"],
    np.nan,
)
comparison_df


## Notes

To reuse this workflow later with a different adaptation geography or asset
class, change:

- `mask_path`
- `mask_value`
- `vulnerability_reduction`
- `target_exposure_name`
- `target_sector_label`
- `paired_public_component_name` or the baseline component rasters combined with the adapted component
